# SAHIBEX — Qwen3-VL Vision Brain (Kaggle GPU)

This is a separate, non-destructive evaluation notebook. It tests Qwen3-VL on a Kaggle GPU; it does not change the Android phone, does not connect back to SAHIBEX, and does not expose a public service. Kaggle Internet and GPU availability depend on account/session settings.

**Model:** `Qwen/Qwen3-VL-2B-Instruct` as the first practical benchmark candidate. A successful load is not proof of accuracy or production readiness.

In [ ]:
import os, sys, subprocess, importlib.util, json, time, platform
print('Python:', sys.version)
print('Platform:', platform.platform())
print('CUDA diagnostic:')
subprocess.run(['bash','-lc','nvidia-smi'], check=False)
if not importlib.util.find_spec('torch'):
    raise RuntimeError('PyTorch is not available in this Kaggle runtime. Select a GPU notebook image, then rerun.')
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is available. Enable a Kaggle GPU accelerator before continuing.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
# Install only the notebook runtime dependencies. Internet must be enabled in Kaggle.
%pip -q install -U transformers accelerate qwen-vl-utils pillow

In [ ]:
import torch, time
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
MODEL_ID = 'Qwen/Qwen3-VL-2B-Instruct'
print('Loading:', MODEL_ID)
started = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=False)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=False
)
model.eval()
print('Model loaded in seconds:', round(time.time()-started, 2))
print('Model device:', next(model.parameters()).device)
print('GPU allocated GiB:', round(torch.cuda.memory_allocated()/1024**3, 2))

In [ ]:
# Create a deterministic synthetic image locally; no personal image is uploaded.
from PIL import Image, ImageDraw
from qwen_vl_utils import process_vision_info
img = Image.new('RGB', (640, 360), 'white')
draw = ImageDraw.Draw(img)
draw.rectangle((40, 40, 250, 250), outline='black', width=5)
draw.ellipse((330, 55, 530, 255), outline='black', width=5)
draw.text((70, 285), 'SQUARE', fill='black')
draw.text((365, 285), 'CIRCLE', fill='black')
messages = [{
  'role': 'user',
  'content': [
    {'type': 'image', 'image': img},
    {'type': 'text', 'text': 'Describe only what is visible. Name the two shapes and the words written below them. If anything is unclear, say so.'}
  ]
}]
prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(text=[prompt], images=image_inputs, videos=video_inputs, padding=True, return_tensors='pt').to(model.device)
started = time.time()
with torch.inference_mode():
  output_ids = model.generate(**inputs, max_new_tokens=128, do_sample=False)
new_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
answer = processor.batch_decode(new_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
elapsed = round(time.time()-started, 2)
print('Vision answer:', answer)
print('Inference seconds:', elapsed)
assert answer, 'FAIL CLOSED: model returned empty content'
print('Synthetic-image smoke test: RESPONSE_RECEIVED; manually inspect answer quality before marking PASS.')

## Safety and integration status
- This notebook is an evaluation harness only; it does not open a public API or tunnel.
- Do not upload private images, credentials, or sensitive data during initial testing.
- A model response is untrusted output; it must not authorize actions or become VERIFIED without independent checks.
- The phone-to-Kaggle connection is intentionally not implemented here. Before that step, design an authenticated, expiring relay; never expose an unauthenticated model server.
- Mark P7 as PARTIAL until the notebook runs successfully, output quality is checked, resource use is recorded, and a safe integration path is tested.